# SHARP on Kaggle
Add the `doppler_traces` dataset to this notebook and turn on Internet and a GPU.

In [ ]:
import glob, os
DATA = os.path.dirname(glob.glob('/kaggle/input/**/S1a', recursive=True)[0])
!git clone -q https://github.com/francescamen/SHARP /kaggle/working/SHARP
%cd /kaggle/working/SHARP/Python_code
!cp -rs {DATA}/S* doppler_traces/
!sed -i "s/rcParams\['text.usetex'\] = 'true'/rcParams['text.usetex'] = False/" plots_utility.py
!sed -i "s/traces_activities\[4\]/traces_activities[3]/" CSI_doppler_plot_activities.py
!sed -i "s/pickle.dump(csi_matrices_set\[ii\], fp)/pickle.dump(csi_matrices_set[ii].astype(np.float32), fp)/" CSI_doppler_create_dataset_train.py
!sed -i "s/pickle.dump(csi_matrices_wind\[ii\], fp)/pickle.dump(csi_matrices_wind[ii].astype(np.float32), fp)/" CSI_doppler_create_dataset_test.py

## Doppler plots

In [ ]:
!python CSI_doppler_plots_antennas.py ./doppler_traces/ S7a 100 1 E,L1,W,R,J1 20000
!python CSI_doppler_plot_activities.py ./doppler_traces/ S7a 100 1 E,L1,W,R,J1 570 1070

## Dataset creation

In [ ]:
!python CSI_doppler_create_dataset_train.py ./doppler_traces/ S1a,S1b,S1c 31 1 340 30 E,L,W,R,J 4

## Train

In [ ]:
!python CSI_network.py ./doppler_traces/ S1a 100 340 1 32 4 single_ant E,L,W,R,J
!rm -rf doppler_traces/S1*/*_antennas_* *_cache_*

## Test and metrics

In [ ]:
for s in ['S2a', 'S2b', 'S3a', 'S4a', 'S4b', 'S5a', 'S6a', 'S6b', 'S7a']:
    !python CSI_doppler_create_dataset_test.py ./doppler_traces/ {s} 31 1 340 30 E,L,W,R,J 4
    !python CSI_network_test.py ./doppler_traces/ {s} 100 340 1 32 4 single_ant E,L,W,R,J
    !python CSI_network_metrics.py complete_different_E,L,W,R,J_{s}_band_80_subband_1 E,L,W,R,J
    !python CSI_network_metrics_plot.py complete_different_E,L,W,R,J_{s}_band_80_subband_1 E,L,W,R,J
    !rm -rf doppler_traces/{s}/*_antennas_* *_cache_*

## Download results

In [ ]:
!zip -qr /kaggle/working/SHARP_results.zip single_ant_E,L,W,R,J_network.h5 outputs plots